# 01 - Fundamentos de AdaBoost: Pesos Adaptativos y Decision Stumps

## Curso de Machine Learning: Modelos de Clasificación

**AdaBoost (Adaptive Boosting)**, formulado por Yoav Freund y Robert Schapire (1997), fue el primer algoritmo de *boosting* práctico y exitoso de la historia del aprendizaje automático (reconocido con el Premio Gödel en 2003). Su premisa central consiste en combinar secuencialmente múltiples clasificadores débiles (*weak learners*, típicamente **Decision Stumps** o árboles de profundidad 1) modificando adaptativamente los **pesos de las muestras**: las instancias mal clasificadas incrementan su peso en la siguiente iteración, obligando al próximo clasificador a concentrarse en los ejemplos más difíciles.

---

### Contenidos
1. ¿Qué es un *Weak Learner* y un *Decision Stump* ($\text{max\_depth}=1$)?
2. Formulación Matemática del Algoritmo AdaBoost Binario ($y_i \in \{-1, +1\}$)
3. Error Ponderado ($\varepsilon_m$), Peso del Clasificador ($\alpha_m$) y Actualización Exponencial de Pesos ($w_i$)
4. Simulación Matemática Paso a Paso desde Cero en NumPy
5. Visualización de la Evolución de los Pesos de las Instancias y Fronteras Secuenciales
6. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_moons, load_breast_cancer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.metrics import accuracy_score

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)
print("Entorno para Fundamentos de AdaBoost listo.")


## 1. Formulación Matemática de AdaBoost

Sea un conjunto de entrenamiento $\{(\mathbf{x}_1, y_1), \dots, (\mathbf{x}_N, y_N)\}$ con etiquetas $y_i \in \{-1, +1\}$.

### Paso 1: Inicialización Uniforme de Pesos
Al inicio ($m=1$), todas las observaciones tienen exactamente la misma importancia:
$$w_i^{(1)} = \frac{1}{N}, \quad \forall i \in \{1, \dots, N\}$$

### Paso 2: Iteración Secuencial ($m = 1, \dots, M$)
1. **Entrenar un clasificador débil** $G_m(\mathbf{x}) \in \{-1, +1\}$ minimizando el error ponderado con los pesos actuales $w_i^{(m)}$.
2. **Calcular la tasa de error ponderada** $\varepsilon_m$ del clasificador $G_m$:
$$\varepsilon_m = \frac{\sum_{i=1}^N w_i^{(m)} \cdot \mathbb{I}(y_i \neq G_m(\mathbf{x}_i))}{\sum_{i=1}^N w_i^{(m)}}$$
3. **Calcular la autoridad (peso o voz) del clasificador** $\alpha_m$ en la votación final (con tasa de aprendizaje $\eta$):
$$\alpha_m = \eta \cdot \frac{1}{2} \ln\left( \frac{1 - \varepsilon_m}{\varepsilon_m} \right)$$
   - Si $\varepsilon_m \to 0$, entonces $\alpha_m \to +\infty$ (el árbol tiene altísimo poder de voto).
   - Si $\varepsilon_m = 0.5$ (azar puro), entonces $\alpha_m = 0$ (no aporta nada).
4. **Actualizar los pesos de las muestras** para la iteración $m+1$ y normalizar para que sumen $1$:
$$w_i^{(m+1)} = \frac{w_i^{(m)} \cdot \exp\left( -\alpha_m y_i G_m(\mathbf{x}_i) \right)}{Z_m}$$
   - Si la muestra fue **mal clasificada**, $y_i G_m(\mathbf{x}_i) = -1 \implies e^{+\alpha_m} > 1$ (**el peso crece exponencialmente**).
   - Si la muestra fue **bien clasificada**, $y_i G_m(\mathbf{x}_i) = +1 \implies e^{-\alpha_m} < 1$ (**el peso disminuye**).

### Paso 3: Regla de Decisión del Ensamble Final
$$G(\mathbf{x}) = \text{sign}\left( \sum_{m=1}^M \alpha_m G_m(\mathbf{x}) \right)$$


In [ ]:
# Curva teorica de la autoridad del clasificador (alpha_m) en funcion del error ponderado (epsilon_m)
eps = np.linspace(0.01, 0.99, 300)
alpha_teorico = 0.5 * np.log((1 - eps) / eps)

plt.figure(figsize=(8.5, 4.5))
plt.plot(eps, alpha_teorico, color='#d95f02', lw=2.5, label=r'$\alpha_m = \frac{1}{2}\ln\left(\frac{1-\varepsilon_m}{\varepsilon_m}\right)$')
plt.axhline(0, color='black', linestyle='--', lw=1)
plt.axvline(0.5, color='gray', linestyle=':', lw=1.5, label='Azar puro (epsilon = 0.5)')
plt.xlabel('Error Ponderado del Clasificador Debil (epsilon_m)', fontweight='bold')
plt.ylabel('Peso / Autoridad del Modelo (alpha_m)', fontweight='bold')
plt.title('Peso de Votacion asignado a cada Arbol segun su Error Ponderado', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()


## 2. Implementación de AdaBoost desde Cero y Evolución de Pesos ($w_i$)

Vamos a programar el bucle exacto de AdaBoost sobre un conjunto bidimensional para observar visualmente cómo cambian los tamaños de los puntos (proporcionales a $w_i^{(m)}$) tras cada *Decision Stump*:


In [ ]:
X_toy, y_toy_01 = make_moons(n_samples=80, noise=0.25, random_state=42)
y_toy = np.where(y_toy_01 == 0, -1, 1) # Etiquetas en {-1, +1}

N = len(y_toy)
w = np.ones(N) / N
stumps, alphas, historial_pesos = [], [], []

for m in range(4):
    historial_pesos.append(w.copy())
    stump = DecisionTreeClassifier(max_depth=1, random_state=42)
    stump.fit(X_toy, y_toy, sample_weight=w)
    pred_m = stump.predict(X_toy)
    
    # Error ponderado
    incorrectos = (pred_m != y_toy)
    eps_m = np.sum(w[incorrectos]) / np.sum(w)
    eps_m = np.clip(eps_m, 1e-10, 1 - 1e-10)
    
    # Autoridad del arbol
    alpha_m = 0.5 * np.log((1 - eps_m) / eps_m)
    
    # Actualizacion exponencial de pesos
    w = w * np.exp(-alpha_m * y_toy * pred_m)
    w = w / np.sum(w) # Normalizacion
    
    stumps.append(stump)
    alphas.append(alpha_m)
    print(f"Iteracion {m+1}: Error ponderado = {eps_m:.4f} | Peso del Stump (alpha_{m+1}) = {alpha_m:.4f} | Max w_i = {w.max():.4f}")


In [ ]:
# Visualizacion de los Decision Stumps y el crecimiento de los pesos en las muestras mal clasificadas
fig, axes = plt.subplots(1, 4, figsize=(17, 4.2), sharey=True)

for m, ax in enumerate(axes):
    DecisionBoundaryDisplay.from_estimator(
        stumps[m], X_toy, cmap='coolwarm', alpha=0.35, ax=ax, response_method='predict'
    )
    tamanos = historial_pesos[m] * 3500 # Escalar pesos para visualizacion
    ax.scatter(X_toy[:, 0], X_toy[:, 1], c=y_toy, cmap='coolwarm', s=tamanos, edgecolors='k', alpha=0.85)
    ax.set_title(f"Iteracion {m+1}\n(alpha={alphas[m]:.2f})", fontweight='bold')
    ax.set_xlabel('Caracteristica 1')
    if m == 0:
        ax.set_ylabel('Caracteristica 2')

plt.suptitle('Evolucion de los Pesos de las Instancias (Tamaño del Punto = Peso w_i)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Conclusiones

1. **Enfoque Adaptativo en Casos Difíciles**: A diferencia del muestreo uniforme de Bagging, AdaBoost altera la distribución de probabilidad empírica de los datos en cada iteración.
2. **Combinación de Fronteras Lineales por Ejes**: Un solo *Decision Stump* (`max_depth=1`) únicamente puede trazar una línea horizontal o vertical en el espacio de características; sin embargo, la combinación lineal ponderada $\sum \alpha_m G_m(\mathbf{x})$ genera fronteras no lineales altamente expresivas.
